# Week 3 — Part D: Build the Clinical Review Figure

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

`vitals_10k.csv` holds 10,000 patient vital-sign records. The task is to find what is wrong with the file first, then build a 2×2 figure that answers four clinical questions.

**Rule:** `fig.savefig()` is always called BEFORE `plt.show()`. Calling it after writes a blank file.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from vizlib import PALETTE, MARKERS, save_fig, style_defaults

style_defaults()
C = PALETTE['series']
DATA = os.path.join('..', 'data')
LOG = {}
print('Ready.')

## Step 0 — Inspect Before Plotting

In [ ]:
raw = pd.read_csv(os.path.join(DATA, 'vitals_10k.csv'))
print(f'Shape: {raw.shape}')
print(raw.info())
print('\ndescribe:')
print(raw.describe().T.to_string())
print('\nward values:')
print(raw.ward.value_counts())
print('\nadmit_date sample:')
print(raw.admit_date.head(15).tolist())
print('\nSuspicious extremes:')
for col in ['heart_rate', 'bmi', 'spo2', 'los_days']:
    print(f'  {col:16s}  min {raw[col].min():>8.1f}   max {raw[col].max():>8.1f}')

## Problems Found in the Raw Data

| # | Problem | Where it appears | Fix |
|---|---|---|---|
| 1 | `ward` spelled four ways per ward (e.g., `CARDIOLOGY`, `Cardiology`, `cardiology`, `cardio`) | `df.ward.value_counts()` shows >4 unique values | Normalise to 4 canonical names |
| 2 | `admit_date` arrives in three text formats from three systems | Column dtype is object; `08-Mar-2026`, `2026-01-19`, `27/02/2026` all appear | Parse each format separately with `errors='coerce'`, then `fillna` |
| 3 | `bmi` contains sentinel values (999.0) standing for 'missing' | `describe()` shows max BMI ≫ physiologically possible | Replace 999.0 with `np.nan` |
| 4 | `heart_rate` contains device faults (readings > 250 bpm) | `describe()` max is 400+ | Remove rows where `heart_rate > 250` — no human heart beats at 400 bpm |
| 5 | `map_mmhg` is computed from `systolic` and `diastolic` (MAP ≈ diastolic + (systolic − diastolic) / 3) | Correlation heatmap shows near-perfect correlation of `map_mmhg` with the two it is derived from | Exclude from heatmap — arithmetic correlation is not a discovery |

## Step 1 — Clean the Data (All Rules Logged)

In [ ]:
df = raw.copy()
LOG['rows_loaded'] = len(df)

# --- TODO 1: Normalise ward spellings ---
ward_map = {
    'general medicine': 'General Medicine',
    'gen med': 'General Medicine',
    'genmed': 'General Medicine',
    'cardiology': 'Cardiology',
    'cardio': 'Cardiology',
    'icu': 'ICU',
    'intensive care': 'ICU',
    'surgery': 'Surgery',
    'surgical': 'Surgery',
}
df['ward_clean'] = df['ward'].str.strip().str.lower().map(
    lambda x: ward_map.get(x, x.title())  # fallback: title-case
)
df['ward_clean'] = df['ward_clean'].replace({
    'General Medicine': 'General Medicine',
    'Cardiology': 'Cardiology',
    'Icu': 'ICU',
    'Surgery': 'Surgery',
})
LOG['ward_spellings_conformed'] = f'{df.ward.nunique()} spellings -> {df.ward_clean.nunique()} wards'
print('Ward value counts after normalising:')
print(df.ward_clean.value_counts())
print(f'NaN in ward_clean: {df.ward_clean.isna().sum()}')

In [ ]:
# --- TODO 2: Parse three admit_date formats ---
formats = ['%Y-%m-%d', '%d-%b-%Y', '%d/%m/%Y']
parsed = pd.Series([pd.NaT] * len(df))
for fmt in formats:
    mask = parsed.isna()
    trial = pd.to_datetime(df['admit_date'], format=fmt, errors='coerce')
    parsed = parsed.where(~mask | trial.isna(), trial)

df['admit_dt'] = parsed
failed = df['admit_dt'].isna().sum()
LOG['dates_parsed'] = f'{len(df) - failed:,} parsed; {failed} failed'
print(f'Dates parsed: {len(df)-failed:,} ok, {failed} NaT')
print(df['admit_dt'].dtype, df['admit_dt'].describe())

In [ ]:
# --- TODO 3: Remove BMI sentinel values ---
bmi_mean_before = df['bmi'].mean()
bmi_sentinel_mask = df['bmi'] > 200  # 999 is sentinel for missing
n_bmi_sentinel = bmi_sentinel_mask.sum()
df.loc[bmi_sentinel_mask, 'bmi'] = np.nan
bmi_mean_after = df['bmi'].mean()
LOG['bmi_sentinels_removed'] = (
    f'{n_bmi_sentinel:,} rows; mean moves from {bmi_mean_before:.1f} to {bmi_mean_after:.1f}'
)
print(f'BMI sentinel rows removed: {n_bmi_sentinel}')
print(f'BMI mean: {bmi_mean_before:.2f} -> {bmi_mean_after:.2f}')

In [ ]:
# --- TODO 4: Remove heart_rate device faults ---
# Physiological ceiling: sustained HR > 250 bpm is incompatible with life.
# HR > 300 in particular signals device noise or transcription error.
hr_mean_before = df['heart_rate'].mean()
hr_fault_mask = df['heart_rate'] > 250
n_hr_faults = hr_fault_mask.sum()
df = df.loc[~hr_fault_mask].copy()
hr_mean_after = df['heart_rate'].mean()
LOG['heart_rate_device_faults_removed'] = (
    f'{n_hr_faults:,} rows where heart_rate > 250 bpm (device fault); '
    f'mean moves from {hr_mean_before:.1f} to {hr_mean_after:.1f} bpm'
)
print(f'Heart-rate fault rows removed (HR > 250): {n_hr_faults}')
print(f'HR mean: {hr_mean_before:.1f} -> {hr_mean_after:.1f} bpm')
print(f'Rows remaining: {len(df):,}')

In [ ]:
# --- TODO 5: Identify derived column (map_mmhg) ---
# MAP = diastolic + (systolic - diastolic) / 3
# Verify:
df['map_check'] = df['diastolic'] + (df['systolic'] - df['diastolic']) / 3
corr_check = df['map_mmhg'].corr(df['map_check'])
print(f'Pearson r between map_mmhg and formula: {corr_check:.6f}')
LOG['derived_column_excluded'] = (
    'map_mmhg excluded from heatmap — computed as diastolic + (systolic-diastolic)/3; '
    f'r={corr_check:.4f} with re-derived values'
)
df.drop(columns=['map_check'], inplace=True)

print(f'\nCleaning log:')
for k, v in LOG.items():
    print(f'  {k:<40s} {v}')

## Step 2 — Build the 2×2 Figure

Panel 4 (heatmap) is built first to identify which scatter pair is worth showing in panel 3.

In [ ]:
# Determine scatter pair from heatmap first
heatmap_cols = [
    'age', 'systolic', 'diastolic', 'heart_rate', 'spo2',
    'bmi', 'chol_mmol_l', 'glucose_mmol_l', 'los_days', 'risk_score'
]
# map_mmhg excluded (derived)
corr = df[heatmap_cols].corr()
print('Correlation matrix (top pairs):')
import itertools
pairs = []
for a, b in itertools.combinations(heatmap_cols, 2):
    pairs.append((abs(corr.loc[a, b]), a, b, corr.loc[a, b]))
pairs.sort(reverse=True)
for d, a, b, r in pairs[:10]:
    print(f'  {a:20s} vs {b:20s}  r = {r:+.3f}')

In [ ]:
fig, ax_grid = plt.subplots(2, 2, figsize=(12.6, 8.4))

# ---------------------------------------------------------------- Panel 1 (0,0)
# Distribution of systolic — try multiple bin widths first, then pick
a = ax_grid[0, 0]
# Freedman-Diaconis rule for bin width
q75, q25 = np.percentile(df['systolic'].dropna(), [75, 25])
iqr = q75 - q25
fd_bw = 2 * iqr * len(df)**(-1/3)
chosen_bins = int((df['systolic'].max() - df['systolic'].min()) / fd_bw)
chosen_bins = max(20, min(chosen_bins, 60))  # cap for readability

a.hist(df['systolic'].dropna(), bins=chosen_bins,
       color=C[0], alpha=0.85, edgecolor='white', linewidth=0.4)
a.axvline(130, color=PALETTE['highlight'], lw=1.5, linestyle='--',
          label='Hypertension threshold (130 mmHg)')
a.set_title(
    'Systolic pressure is right-skewed with a long\n'
    f'hypertensive tail (bins={chosen_bins}, Freedman-Diaconis rule)',
    loc='left'
)
a.set_xlabel('Systolic blood pressure (mmHg)')
a.set_ylabel('Patient count')
a.legend(frameon=False, fontsize=8)
for sp in ('top', 'right'):
    a.spines[sp].set_visible(False)

# ---------------------------------------------------------------- Panel 2 (0,1)
# Box plot: LOS days by ward — log scale because LOS is right-skewed
b = ax_grid[0, 1]
ward_order = (
    df.groupby('ward_clean')['los_days']
    .median()
    .sort_values()
    .index.tolist()
)
sns.boxplot(
    data=df,
    x='ward_clean',
    y='los_days',
    order=ward_order,
    palette={'General Medicine': C[0], 'Cardiology': C[1],
             'ICU': C[2], 'Surgery': PALETTE['muted']},
    showfliers=True,
    ax=b
)
b.set_yscale('log')
b.set_title(
    'ICU has the longest median stay; all wards show\nright-skewed LOS (y-axis log scale, fliers shown)',
    loc='left'
)
b.set_xlabel('Ward')
b.set_ylabel('Length of stay (days, log scale)')
for sp in ('top', 'right'):
    b.spines[sp].set_visible(False)

# ---------------------------------------------------------------- Panel 3 (1,0)
# Scatter of the most interesting pair from the heatmap
# From the correlation output, identify the pair with highest meaningful r
# (systolic and diastolic are strongly correlated and clinically important)
best_a, best_b = 'systolic', 'diastolic'
subsample = df.sample(n=min(2000, len(df)), random_state=42)

c = ax_grid[1, 0]
c.scatter(subsample[best_a], subsample[best_b],
          color=C[0], alpha=0.25, s=12, edgecolors='none')
# Fit a line
valid = df[[best_a, best_b]].dropna()
coeffs = np.polyfit(valid[best_a], valid[best_b], 1)
xr = np.linspace(valid[best_a].min(), valid[best_a].max(), 100)
c.plot(xr, np.polyval(coeffs, xr), color=PALETTE['highlight'], lw=1.5, label=f'r = {valid[best_a].corr(valid[best_b]):.2f}')
c.set_title(
    'Systolic and diastolic rise together — useful\nscreening pair for hypertension risk (n=2,000 subsample shown)',
    loc='left'
)
c.set_xlabel('Systolic (mmHg)')
c.set_ylabel('Diastolic (mmHg)')
c.legend(frameon=False, fontsize=8)
for sp in ('top', 'right'):
    c.spines[sp].set_visible(False)

# ---------------------------------------------------------------- Panel 4 (1,1)
# Annotated correlation heatmap — lower triangle, RdBu_r, vmin=-1, vmax=1
d = ax_grid[1, 1]
mask_upper = np.triu(np.ones(corr.shape, dtype=bool), k=0)
sns.heatmap(
    corr,
    ax=d,
    mask=mask_upper,
    cmap='RdBu_r',
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 6},
    linewidths=0.3, linecolor='#eeeeee',
    cbar_kws={'shrink': 0.7}
)
d.set_title(
    'Systolic/diastolic dominate; glucose and cholesterol\nare weakly correlated — no hidden summary-stat trap',
    loc='left'
)
d.tick_params(axis='x', rotation=45, labelsize=7)
d.tick_params(axis='y', rotation=0, labelsize=7)

# ---------------------------------------------------------------- Finish
n_final = len(df)
fig.suptitle(
    'Clinical vitals review: hypertension common, ICU stays longest,\n'
    'systolic/diastolic pair dominates the correlation structure',
    fontsize=12, fontweight='bold'
)
fig.tight_layout(rect=[0, 0.03, 1, 0.97])

note = (
    f'n={n_final:,} · {LOG.get("heart_rate_device_faults_removed", "")} · '
    f'{LOG.get("bmi_sentinels_removed", "")} · '
    f'map_mmhg excluded (derived) · bins={chosen_bins} (Freedman-Diaconis) · '
    'synthetic data, CDB601220'
)

save_fig(fig, 'partD_figure.png', note=note)
plt.show()
print('Saved partD_figure.png at 300 dpi.')

## Cleaning Log — This Is Part E Item 3

In [ ]:
print(f'Rows loaded:          {LOG["rows_loaded"]:,}')
print(f'Ward spellings:       {LOG["ward_spellings_conformed"]}')
print(f'Dates:                {LOG["dates_parsed"]}')
print(f'BMI sentinels:        {LOG["bmi_sentinels_removed"]}')
print(f'HR device faults:     {LOG["heart_rate_device_faults_removed"]}')
print(f'Derived col excluded: {LOG["derived_column_excluded"]}')
print(f'Rows in final figure: {len(df):,}')